# Still-life loop — Runware mainframes + FlowMorph bridge schedules

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MNoichl/FluxFlowMorph/blob/main/notebooks/StillLife_Loop_Runware_Art.ipynb)

This local-only notebook reads a validated art-loop JSON plan, generates ordered mainframes, then runs one FlowMorph transition for every adjacent pair. Mainframes after the first receive a deliberately faint, blurred version of the previous image as reference conditioning. This gives weak compositional continuity without asking the model to copy the preceding painting.

Each transition has exactly 20 ordered prompts. The first and last align with the endpoint concepts; the exact endpoint images replace those two display slots, while prompts 2–19 visibly steer the interior morph frames. The public Runware mirror is anonymous: no Hugging Face token is needed.

The notebook is tracked on GitHub and can be launched with the Colab badge above. Project prompt JSONs remain ignored, except for the tracked example schema.

## 1. Editable settings

These are the main artistic and numerical controls. Their defaults reproduce the settings used by the current prompt plan. FLUX.2 Klein does not expose a conventional img2img denoising `strength`; `MAINFRAME_REFERENCE_STRENGTH` is the active equivalent here—the fraction of the blurred previous mainframe mixed into the next reference image. Higher values produce stronger visual continuity.

When the VS Code notebook is attached to a Colab kernel, paths refer to the remote `/content` filesystem. Upload custom JSON files to `/content/prompts`; the loader also checks `/content` and the repository prompt directory. Set `TRANSITION_INDICES = [0]` for one test transition or `None` for the complete loop.

In [ ]:
PROJECT_ROOT = "/content/FlowMorphKlein9B"
REPOSITORY_URL = "https://github.com/MNoichl/FluxFlowMorph.git"
UPDATE_REPOSITORY = True
CONFIG_PATH = f"{PROJECT_ROOT}/configs/full_9b_lora.yaml"
PROMPT_SPEC_FILENAME = "sciences_red_path_baroque_loop_v2.json"
PROMPT_DIRECTORY = "/content/prompts"  # Colab/VS Code upload destination.
PROMPT_SPEC_PATH = f"{PROMPT_DIRECTORY}/{PROMPT_SPEC_FILENAME}"
ASSET_ROOT = "/content/flowmorph_art_loops"
HF_CACHE_DIR = "/content/hf_cache"
MOUNT_DRIVE = True  # Change to True to persist every completed stage immediately.
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/FluxFlowMorphArt"  # Created if missing.
PROFILE = "auto"

# Mainframe generation controls (current values)
MAINFRAME_WIDTH = 1024   # 256–2048, divisible by 16; larger images require much more VRAM.
MAINFRAME_HEIGHT = 1024  # 256–2048, divisible by 16.
MAINFRAME_LORA_SCALE = 1.0
MAINFRAME_INFERENCE_STEPS = 28
MAINFRAME_GUIDANCE_SCALE = 4.0
MAINFRAME_SEED = 1729
MAINFRAME_CONTINUITY_ENABLED = True
MAINFRAME_REFERENCE_STRENGTH = 0.14  # Previous-image blend; valid range: 0 < value <= 0.35.
MAINFRAME_REFERENCE_BLUR = 14.0       # Higher values retain broad layout but suppress details.
MAINFRAME_REFERENCE_BACKGROUND = (116, 105, 91)

# FlowMorph fitting and rendering controls (current values)
FLOWMORPH_FIT_LORA_SCALE = 1.2
FLOWMORPH_RENDER_LORA_SCALE = 1.2
FLOWMORPH_GUIDANCE_SCALE = 3.6
FLOWMORPH_SCHEDULER_POINTS = 100
FLOWMORPH_START_TIMESTEP_INDEX = 35
FLOWMORPH_SOURCE_OPTIMIZATION_STEPS = 100
FLOWMORPH_TARGET_OPTIMIZATION_STEPS = 100
FLOWMORPH_PRED_LEARNING_RATE = 0.04
FLOWMORPH_U_LEARNING_RATE = 0.01
FLOWMORPH_RENDER_INDICES =  [*range(35, 100, 5), 99] #[35, 55, 75, 95]  # Fast/sparse. Try [*range(35, 100, 5), 99] for cleaner frames.
FLOWMORPH_ALPHA_CURVE = "smootherstep"  # "linear", "smoothstep", "smootherstep", or "sigmoid".
FLOWMORPH_ALPHA_SHARPNESS = 6.0  # Used only by the sigmoid curve; higher crosses the midpoint faster.
FLOWMORPH_CHECKPOINT_EVERY = 25
OUTPUT_FPS = 12

# Run selection and output controls
RUN_TRIAL_KEYFRAME = True
TRIAL_KEYFRAME_INDEX = None  # None chooses a random mainframe prompt; or set 0–19.
TRIAL_SEED = None  # None chooses a fresh random seed; set an integer for controlled comparisons.
TRIAL_DISPLAY_MAX_WIDTH = 768
CONTACT_SHEET_DISPLAY_MAX_WIDTH = 1000
TRANSITION_PREVIEW_DISPLAY_WIDTH = 768
LOOP_AUTO_ROTATE_TO_QUIETEST_CUT = True  # Makes the playback boundary fall on the least-changing adjacent pair.
LOOP_SEAM_AUDIT_SIZE = 192  # Downsample size used only for fast seam measurements.
LOOP_SEAM_DISPLAY_MAX_WIDTH = 1000

# Final RIFE + circular SSIM video post-processing
RUN_RIFE_POSTPROCESS = True
RIFE_REPOSITORY_URL = "https://github.com/hzwer/Practical-RIFE.git"
RIFE_REPOSITORY_REVISION = "17d8c7a1005b37f4c97bfee04e316aaec7fdc536"
RIFE_ROOT = "/content/Practical-RIFE"
RIFE_MODEL_REPOSITORY = "Bash2X/RIFE-Models"
RIFE_MODEL_REVISION = "feaf6d11238b4a1e9f015a5d18c18df152affd20"
RIFE_MODEL_FILENAME = "RIFE_v4.25.zip"
RIFE_MULTIPLIER = 4  # Dense temporal lattice; 4 is usually enough for the already-similar FlowMorph frames.
RIFE_SCALE = 1.0  # Use 0.5 only if interpolation runs out of VRAM.
RIFE_USE_FP16 = True
RIFE_FINAL_FPS = 24.0
RIFE_SSIM_ANALYSIS_SIZE = 192
RIFE_SSIM_WEIGHT_FLOOR = 1e-6
RIFE_VIDEO_CRF = 16
RIFE_KEEP_WORK_FRAMES = False  # Dense PNGs can occupy several GB; final video and diagnostics are always kept.
RIFE_DISPLAY_WIDTH = 768
REGENERATE_MAINFRAMES = True
TRANSITION_INDICES = None  # Use [0] for a first test; None renders every transition.
DOWNLOAD_FINAL_PREVIEW = False


## 2. GPU, repository, and dependencies

In [ ]:
import platform
import subprocess
import sys
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for the FLUX.2 Klein 9B workflow.")
properties = torch.cuda.get_device_properties(0)
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
print({
    "python": sys.version,
    "platform": platform.platform(),
    "gpu": properties.name,
    "total_vram_gib": total_bytes / 2**30,
    "free_vram_gib": free_bytes / 2**30,
    "bf16": torch.cuda.is_bf16_supported(),
})


In [ ]:
import importlib.metadata as importlib_metadata
import json
import re

project_path = Path(PROJECT_ROOT)
if not (project_path / "pyproject.toml").is_file():
    subprocess.check_call(["git", "clone", "--depth", "1", REPOSITORY_URL, PROJECT_ROOT])
elif UPDATE_REPOSITORY:
    subprocess.check_call(["git", "-C", PROJECT_ROOT, "pull", "--ff-only"])
requirements_path = project_path / "requirements-colab.txt"
requirements_text = requirements_path.read_text(encoding="utf-8")
pinned_versions = dict(re.findall(r"(?m)^([A-Za-z0-9_.-]+)==([^\s;]+)$", requirements_text))
dependency_issues = []
for distribution_name, expected_version in pinned_versions.items():
    try:
        installed_version = importlib_metadata.version(distribution_name)
    except importlib_metadata.PackageNotFoundError:
        dependency_issues.append(f"{distribution_name} is missing")
    else:
        if installed_version != expected_version:
            dependency_issues.append(
                f"{distribution_name} is {installed_version}, expected {expected_version}"
            )
try:
    installed_diffusers_version = importlib_metadata.version("diffusers")
except importlib_metadata.PackageNotFoundError:
    dependency_issues.append("diffusers is missing")
else:
    if installed_diffusers_version != "0.39.0":
        dependency_issues.append(
            f"diffusers is {installed_diffusers_version}, expected API-compatible 0.39.0"
        )
# Test imports in a fresh Python process. This checks the files on disk without
# being confused by binary modules already cached in this notebook kernel.
disk_probe = subprocess.run(
    [
        sys.executable,
        "-c",
        "import numpy, scipy, transformers; from diffusers import Flux2KleinPipeline",
    ],
    capture_output=True,
    text=True,
)
if disk_probe.returncode != 0:
    dependency_issues.append("clean-process import probe failed")

if dependency_issues:
    print("Installing dependencies because:")
    for issue in dependency_issues:
        print(" -", issue)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(requirements_path)])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", PROJECT_ROOT])
    raise RuntimeError(
        "Dependencies were installed successfully. Restart the notebook kernel once, "
        "then rerun from the first configuration cell; the installer will be skipped."
    )

print("Pinned dependencies already match; skipping pip installation.")
try:
    import numpy
    import scipy
    from diffusers import Flux2KleinPipeline as _Flux2KleinPipelineImportCheck
except Exception as exc:
    raise RuntimeError(
        "The installed packages are healthy, but this kernel still has stale binary modules "
        "in memory. Restart the kernel without reinstalling, then rerun the notebook. "
        f"Original import error: {exc}"
    ) from exc

# Make the source importable immediately in this kernel, even when Colab/VS Code
# does not refresh editable installs until the Python process is restarted.
import importlib
package_source = str(Path(PROJECT_ROOT) / "src")
if package_source not in sys.path:
    sys.path.insert(0, package_source)
importlib.invalidate_caches()
import flowmorph_klein

project_commit = subprocess.check_output(["git", "-C", PROJECT_ROOT, "rev-parse", "HEAD"], text=True).strip()
print("Repository and compatible dependencies ready at commit:", project_commit)
print("FlowMorph import ready from:", flowmorph_klein.__file__)


## 3. Load and validate the prompt plan

Validation fails before model loading if IDs, transition order, loop closure, model revision, resolution, or the 20-prompt counts are wrong.

In [ ]:
!ls 

In [ ]:
import importlib
import sys
from pathlib import Path

# Self-heal after a kernel restart or an editable-install refresh failure.
package_source = Path(PROJECT_ROOT) / "src"
package_init = package_source / "flowmorph_klein" / "__init__.py"
if not package_init.is_file():
    raise FileNotFoundError(
        f"Repository source is missing at {package_init}. Run the repository/setup cell above first."
    )
if str(package_source) not in sys.path:
    sys.path.insert(0, str(package_source))
importlib.invalidate_caches()

from flowmorph_klein.art_loop import load_art_loop_spec, safe_transition_name

# Prefer the upload directory, but also recognize files uploaded directly to
# /content and prompt files stored inside the cloned repository.
prompt_candidates = [
    Path(PROMPT_SPEC_PATH),
    Path("/content") / PROMPT_SPEC_FILENAME,
    Path(PROJECT_ROOT) / "art_projects" / "prompts" / PROMPT_SPEC_FILENAME,
    Path.cwd() / "prompts" / PROMPT_SPEC_FILENAME,
    Path.cwd() / PROMPT_SPEC_FILENAME,
]
prompt_spec = next((candidate for candidate in prompt_candidates if candidate.is_file()), None)
if prompt_spec is None:
    raise FileNotFoundError(
        "Prompt JSON was not found. Upload it to /content/prompts or set "
        f"PROMPT_SPEC_PATH explicitly. Checked: {[str(path) for path in prompt_candidates]}"
    )
print("Loading prompt plan from:", prompt_spec)
SPEC = load_art_loop_spec(prompt_spec)
if not SPEC.project.loop:
    raise ValueError("This notebook requires project.loop=true for exact cyclic assembly.")
mainframe_prompt_by_id = {item.id: item.prompt for item in SPEC.mainframes}
for transition in SPEC.transitions:
    if transition.bridge_prompts[0] != mainframe_prompt_by_id[transition.from_id]:
        raise ValueError(
            f"First bridge prompt for {transition.from_id}->{transition.to_id} must exactly equal its source prompt"
        )
    if transition.bridge_prompts[-1] != mainframe_prompt_by_id[transition.to_id]:
        raise ValueError(
            f"Last bridge prompt for {transition.from_id}->{transition.to_id} must exactly equal its target prompt"
        )
closing_transition = SPEC.transitions[-1]
if closing_transition.to_id != SPEC.mainframes[0].id:
    raise ValueError("The final transition must return to the first mainframe.")
print("Cyclic prompt contract verified: exact textual endpoints and final-to-first closure.")
if not (256 <= MAINFRAME_WIDTH <= 2048 and MAINFRAME_WIDTH % 16 == 0):
    raise ValueError("MAINFRAME_WIDTH must be 256–2048 and divisible by 16")
if not (256 <= MAINFRAME_HEIGHT <= 2048 and MAINFRAME_HEIGHT % 16 == 0):
    raise ValueError("MAINFRAME_HEIGHT must be 256–2048 and divisible by 16")
if not (1 <= MAINFRAME_INFERENCE_STEPS <= 100):
    raise ValueError("MAINFRAME_INFERENCE_STEPS must be between 1 and 100")
if not (0.0 <= MAINFRAME_GUIDANCE_SCALE <= 20.0):
    raise ValueError("MAINFRAME_GUIDANCE_SCALE must be between 0 and 20")
if not (0.0 < MAINFRAME_LORA_SCALE <= 4.0):
    raise ValueError("MAINFRAME_LORA_SCALE must lie in (0, 4]")
if not (0.0 < MAINFRAME_REFERENCE_STRENGTH <= 0.35):
    raise ValueError("MAINFRAME_REFERENCE_STRENGTH must lie in (0, 0.35]")
if MAINFRAME_REFERENCE_BLUR < 0.0:
    raise ValueError("MAINFRAME_REFERENCE_BLUR cannot be negative")
if len(MAINFRAME_REFERENCE_BACKGROUND) != 3 or any(
    channel < 0 or channel > 255 for channel in MAINFRAME_REFERENCE_BACKGROUND
):
    raise ValueError("MAINFRAME_REFERENCE_BACKGROUND must contain three values in [0, 255]")
continuity_config = SPEC.generation.continuity.model_copy(update={
    "enabled": MAINFRAME_CONTINUITY_ENABLED,
    "reference_blend": MAINFRAME_REFERENCE_STRENGTH,
    "blur_radius": MAINFRAME_REFERENCE_BLUR,
    "background_rgb": tuple(MAINFRAME_REFERENCE_BACKGROUND),
})
generation_config = SPEC.generation.model_copy(update={
    "width": MAINFRAME_WIDTH,
    "height": MAINFRAME_HEIGHT,
    "num_inference_steps": MAINFRAME_INFERENCE_STEPS,
    "guidance_scale": MAINFRAME_GUIDANCE_SCALE,
    "seed": MAINFRAME_SEED,
    "continuity": continuity_config,
})
lora_config = SPEC.lora.model_copy(update={"scale": MAINFRAME_LORA_SCALE})
flowmorph_config = SPEC.flowmorph.model_copy(update={"fps": float(OUTPUT_FPS)})
SPEC = SPEC.model_copy(update={
    "generation": generation_config,
    "lora": lora_config,
    "flowmorph": flowmorph_config,
})
PROJECT_OUTPUT = Path(ASSET_ROOT) / SPEC.project.name
MAINFRAME_DIRECTORY = PROJECT_OUTPUT / "mainframes"
TRANSITION_ROOT = PROJECT_OUTPUT / "transitions"
for directory in (PROJECT_OUTPUT, MAINFRAME_DIRECTORY, TRANSITION_ROOT, Path(HF_CACHE_DIR)):
    directory.mkdir(parents=True, exist_ok=True)
print({
    "project": SPEC.project.name,
    "closed_loop": SPEC.project.loop,
    "mainframes": [item.id for item in SPEC.mainframes],
    "transitions": [safe_transition_name(item.from_id, item.to_id) for item in SPEC.transitions],
    "prompts_per_transition": SPEC.flowmorph.frame_count,
    "mainframe_lora_scale": SPEC.lora.scale,
    "mainframe_reference_strength": SPEC.generation.continuity.reference_blend,
    "mainframe_reference_blur": SPEC.generation.continuity.blur_radius,
    "mainframe_steps": SPEC.generation.num_inference_steps,
    "mainframe_guidance": SPEC.generation.guidance_scale,
    "image_size": [SPEC.generation.width, SPEC.generation.height],
    "output_fps": SPEC.flowmorph.fps,
})


In [ ]:
DRIVE_ENABLED = False
if MOUNT_DRIVE:
    try:
        from google.colab import drive
    except ImportError as error:
        raise RuntimeError("Drive mounting requires a Google Colab kernel.") from error
    drive.mount("/content/drive")
    Path(DRIVE_OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
    DRIVE_ENABLED = True
    print("Drive persistence enabled. Base directory:", Path(DRIVE_OUTPUT_ROOT).resolve())
    print("Every completed mainframe batch, transition, and final loop will be copied before work continues.")
else:
    print("Drive persistence disabled. Set MOUNT_DRIVE=True in section 1 to enable it.")


## 4. Generate softly related mainframes

The first painting is text-to-image. For every later painting, `make_soft_reference` blurs the preceding image and mixes only `MAINFRAME_REFERENCE_BLEND` of it into a neutral field before passing it as the reference image. Tune the blend, blur, image size, LoRA scale, guidance, steps, and related controls in section 1, then rerun the trial cell before generating the full sequence.

In [ ]:
from flowmorph_klein.environment import AuthenticationResult, verify_model_access

ANONYMOUS_ACCESS = AuthenticationResult(token=None, source="anonymous")
MODEL_ACCESS = verify_model_access(
    ANONYMOUS_ACCESS,
    model_id=SPEC.generation.model_id,
    revision=SPEC.generation.model_revision,
)
print(MODEL_ACCESS)


### 4a. Trial one independent keyframe

Run this cell repeatedly while tuning LoRA scale, guidance, inference steps, dimensions, prompt selection, and seed. It generates one text-to-image trial without continuity conditioning. The loaded pipeline is retained and reused by the full mainframe cell below.

In [ ]:
import gc
import json
import os
import random
import shutil
from datetime import datetime, timezone
from diffusers import Flux2KleinPipeline
from flowmorph_klein.art_loop import apply_prompt_prefix, persist_artifact_tree
from flowmorph_klein.lora import load_flux2_lora
from huggingface_hub import hf_hub_download
from IPython.display import Markdown, display
from PIL import Image

def _sync_spec_with_current_settings():
    global SPEC
    if not (256 <= MAINFRAME_WIDTH <= 2048 and MAINFRAME_WIDTH % 16 == 0):
        raise ValueError("MAINFRAME_WIDTH must be 256–2048 and divisible by 16")
    if not (256 <= MAINFRAME_HEIGHT <= 2048 and MAINFRAME_HEIGHT % 16 == 0):
        raise ValueError("MAINFRAME_HEIGHT must be 256–2048 and divisible by 16")
    if not (1 <= MAINFRAME_INFERENCE_STEPS <= 100):
        raise ValueError("MAINFRAME_INFERENCE_STEPS must be between 1 and 100")
    if not (0.0 <= MAINFRAME_GUIDANCE_SCALE <= 20.0):
        raise ValueError("MAINFRAME_GUIDANCE_SCALE must be between 0 and 20")
    if not (0.0 < MAINFRAME_LORA_SCALE <= 4.0):
        raise ValueError("MAINFRAME_LORA_SCALE must lie in (0, 4]")
    if not (0.0 < MAINFRAME_REFERENCE_STRENGTH <= 0.35):
        raise ValueError("MAINFRAME_REFERENCE_STRENGTH must lie in (0, 0.35]")
    if not (0.0 <= MAINFRAME_REFERENCE_BLUR <= 64.0):
        raise ValueError("MAINFRAME_REFERENCE_BLUR must be between 0 and 64")
    if len(MAINFRAME_REFERENCE_BACKGROUND) != 3 or any(
        not 0 <= channel <= 255 for channel in MAINFRAME_REFERENCE_BACKGROUND
    ):
        raise ValueError("MAINFRAME_REFERENCE_BACKGROUND needs three channels in [0, 255]")
    if MAINFRAME_SEED < 0:
        raise ValueError("MAINFRAME_SEED cannot be negative")
    continuity = SPEC.generation.continuity.model_copy(update={
        "enabled": MAINFRAME_CONTINUITY_ENABLED,
        "reference_blend": MAINFRAME_REFERENCE_STRENGTH,
        "blur_radius": MAINFRAME_REFERENCE_BLUR,
        "background_rgb": tuple(MAINFRAME_REFERENCE_BACKGROUND),
    })
    generation = SPEC.generation.model_copy(update={
        "width": MAINFRAME_WIDTH,
        "height": MAINFRAME_HEIGHT,
        "num_inference_steps": MAINFRAME_INFERENCE_STEPS,
        "guidance_scale": MAINFRAME_GUIDANCE_SCALE,
        "seed": MAINFRAME_SEED,
        "continuity": continuity,
    })
    SPEC = SPEC.model_copy(update={
        "generation": generation,
        "lora": SPEC.lora.model_copy(update={"scale": MAINFRAME_LORA_SCALE}),
        "flowmorph": SPEC.flowmorph.model_copy(update={"fps": float(OUTPUT_FPS)}),
    })

_sync_spec_with_current_settings()

try:
    import peft.tuners.lora.torchao as peft_torchao_dispatch
except ImportError:
    peft_torchao_dispatch = None
else:
    peft_torchao_dispatch.is_torchao_available = lambda: False

downloaded_lora = Path(hf_hub_download(
    repo_id=SPEC.lora.source,
    filename=SPEC.lora.weight_name,
    revision=SPEC.lora.revision,
    cache_dir=HF_CACHE_DIR,
))
lora_stage_directory = Path(HF_CACHE_DIR) / "flowmorph_lora_files" / SPEC.lora.revision[:12]
lora_stage_directory.mkdir(parents=True, exist_ok=True)
LOCAL_LORA_PATH = lora_stage_directory / SPEC.lora.weight_name
source_blob = downloaded_lora.resolve()
if not LOCAL_LORA_PATH.is_file():
    try:
        os.link(source_blob, LOCAL_LORA_PATH)
    except OSError:
        shutil.copy2(source_blob, LOCAL_LORA_PATH)
if LOCAL_LORA_PATH.stat().st_size != source_blob.stat().st_size:
    raise RuntimeError(f"Staged LoRA size mismatch at {LOCAL_LORA_PATH}")

def _release_mainframe_pipeline():
    previous = globals().pop("MAINFRAME_PIPE", None)
    globals().pop("MAINFRAME_PIPE_LORA_SCALE", None)
    if previous is not None:
        maybe_free = getattr(previous, "maybe_free_model_hooks", None)
        if callable(maybe_free):
            maybe_free()
        del previous
        gc.collect()
        torch.cuda.empty_cache()

def _load_mainframe_pipeline():
    pipeline = Flux2KleinPipeline.from_pretrained(
        SPEC.generation.model_id,
        revision=SPEC.generation.model_revision,
        cache_dir=HF_CACHE_DIR,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    )
    report = load_flux2_lora(
        pipeline,
        str(LOCAL_LORA_PATH),
        adapter_name=SPEC.lora.adapter_name,
        scale=SPEC.lora.scale,
        require_base_9b_provenance=False,
        allow_distilled_9b=SPEC.lora.allow_distilled_9b_provenance,
    )
    # PEFT LoRA matrices can remain on CPU while Accelerate moves a parent
    # transformer block to CUDA on a later pipeline call. Fuse the selected
    # trial scale into the frozen transformer and remove runtime adapter
    # matrices before enabling CPU offload. This preserves the LoRA effect
    # while making repeated keyframe calls device-safe.
    fuse_lora = getattr(pipeline, "fuse_lora", None)
    unload_lora = getattr(pipeline, "unload_lora_weights", None)
    if not callable(fuse_lora) or not callable(unload_lora):
        raise RuntimeError("The installed Diffusers build lacks required LoRA fusion APIs")
    fuse_lora(
        components=["transformer"],
        lora_scale=1.0,
        safe_fusing=True,
        adapter_names=[SPEC.lora.adapter_name],
    )
    unload_lora()
    remaining_runtime_lora_parameters = [
        name for name, _ in pipeline.transformer.named_parameters()
        if "lora_" in name.casefold() or ".lora" in name.casefold()
    ]
    if remaining_runtime_lora_parameters:
        raise RuntimeError(
            "LoRA fusion left runtime adapter parameters attached: "
            + ", ".join(remaining_runtime_lora_parameters[:5])
        )
    pipeline.enable_model_cpu_offload()
    pipeline.vae.enable_slicing()
    pipeline.vae.enable_tiling()
    return pipeline, report

TRIAL_KEYFRAME_PATH = None
if RUN_TRIAL_KEYFRAME:
    if TRIAL_KEYFRAME_INDEX is not None and not 0 <= TRIAL_KEYFRAME_INDEX < len(SPEC.mainframes):
        raise IndexError(f"TRIAL_KEYFRAME_INDEX must be between 0 and {len(SPEC.mainframes) - 1}")
    system_random = random.SystemRandom()
    trial_index = (
        TRIAL_KEYFRAME_INDEX
        if TRIAL_KEYFRAME_INDEX is not None
        else system_random.randrange(len(SPEC.mainframes))
    )
    trial_seed = TRIAL_SEED if TRIAL_SEED is not None else system_random.randrange(0, 2**31)
    trial_mainframe = SPEC.mainframes[trial_index]
    trial_prompt = apply_prompt_prefix(SPEC.generation.prompt_prefix, trial_mainframe.prompt)
    loaded_lora_scale = globals().get("MAINFRAME_PIPE_LORA_SCALE")
    if "MAINFRAME_PIPE" in globals() and loaded_lora_scale != float(SPEC.lora.scale):
        print("LoRA scale changed; rebuilding the fused trial pipeline.")
        _release_mainframe_pipeline()
    if "MAINFRAME_PIPE" not in globals():
        MAINFRAME_PIPE, MAINFRAME_LORA_REPORT = _load_mainframe_pipeline()
        MAINFRAME_PIPE_LORA_SCALE = float(SPEC.lora.scale)
        print("Loaded a device-safe fused-LoRA pipeline; same-scale trials will reuse it.")
    else:
        print("Reusing the fused pipeline at the current LoRA scale.")
    trial_result = MAINFRAME_PIPE(
        prompt=trial_prompt,
        height=SPEC.generation.height,
        width=SPEC.generation.width,
        num_inference_steps=SPEC.generation.num_inference_steps,
        guidance_scale=SPEC.generation.guidance_scale,
        generator=torch.Generator(device="cuda").manual_seed(trial_seed),
        output_type="pil",
    )
    trial_image = trial_result.images[0].convert("RGB")
    trial_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    trial_directory = PROJECT_OUTPUT / "trials" / f"{trial_stamp}_{trial_mainframe.id}_seed_{trial_seed}"
    trial_directory.mkdir(parents=True, exist_ok=False)
    TRIAL_KEYFRAME_PATH = trial_directory / "trial.png"
    trial_image.save(TRIAL_KEYFRAME_PATH)
    (trial_directory / "settings.json").write_text(json.dumps({
        "prompt_id": trial_mainframe.id,
        "prompt": trial_prompt,
        "seed": trial_seed,
        "lora_scale": SPEC.lora.scale,
        "guidance_scale": SPEC.generation.guidance_scale,
        "inference_steps": SPEC.generation.num_inference_steps,
        "width": SPEC.generation.width,
        "height": SPEC.generation.height,
    }, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    if DRIVE_ENABLED:
        trial_drive_report = persist_artifact_tree(
            trial_directory,
            DRIVE_OUTPUT_ROOT,
            project_name=SPEC.project.name,
            label="trial_keyframe",
        )
        print("Trial immediately copied to Drive:", trial_drive_report.destination)
    display(Markdown(f"### Trial: `{trial_mainframe.id}`"))
    print({
        "path": str(TRIAL_KEYFRAME_PATH),
        "prompt_index": trial_index,
        "seed": trial_seed,
        "lora_scale": SPEC.lora.scale,
        "guidance_scale": SPEC.generation.guidance_scale,
        "inference_steps": SPEC.generation.num_inference_steps,
        "size": [SPEC.generation.width, SPEC.generation.height],
    })
    trial_preview = trial_image.copy()
    trial_preview.thumbnail((TRIAL_DISPLAY_MAX_WIDTH, TRIAL_DISPLAY_MAX_WIDTH))
    display(trial_preview)
    del trial_result, trial_image, trial_preview
else:
    print("Trial skipped. Set RUN_TRIAL_KEYFRAME=True to generate one tuning image.")


### 4b. Generate the softly related keyframe sequence

In [ ]:
import gc
import json
from flowmorph_klein.art_loop import (
    GeneratedMainframe,
    apply_prompt_prefix,
    generate_mainframes,
    persist_artifact_tree,
)

if "_load_mainframe_pipeline" not in globals():
    raise RuntimeError("Run the trial/setup cell above before generating keyframes.")

if REGENERATE_MAINFRAMES:
    if (
        "MAINFRAME_PIPE" in globals()
        and globals().get("MAINFRAME_PIPE_LORA_SCALE") != float(SPEC.lora.scale)
    ):
        print("Discarding a stale or differently scaled trial pipeline before the full sequence.")
        _release_mainframe_pipeline()
    if "MAINFRAME_PIPE" not in globals():
        MAINFRAME_PIPE, MAINFRAME_LORA_REPORT = _load_mainframe_pipeline()
        MAINFRAME_PIPE_LORA_SCALE = float(SPEC.lora.scale)
        print("Loaded mainframe pipeline for the full keyframe sequence.")
    else:
        print("Reusing the pipeline and LoRA already loaded by the trial cell.")
    MAINFRAME_RECORDS = generate_mainframes(
        MAINFRAME_PIPE, SPEC, MAINFRAME_DIRECTORY, generator_device="cuda"
    )
    _release_mainframe_pipeline()
else:
    MAINFRAME_RECORDS = tuple(
        GeneratedMainframe(
            id=item.id,
            prompt=apply_prompt_prefix(SPEC.generation.prompt_prefix, item.prompt),
            seed=SPEC.generation.seed + item.seed_offset,
            path=MAINFRAME_DIRECTORY / f"mainframe_{index:03d}_{item.id}.png",
            soft_reference_path=None,
        )
        for index, item in enumerate(SPEC.mainframes)
    )
    missing = [str(record.path) for record in MAINFRAME_RECORDS if not record.path.is_file()]
    if missing:
        raise FileNotFoundError("Missing existing mainframes: " + ", ".join(missing))
validated_spec_path = MAINFRAME_DIRECTORY / "art_loop_spec.validated.json"
validated_spec_path.write_text(
    json.dumps(SPEC.model_dump(mode="json", by_alias=True), indent=2, ensure_ascii=False),
    encoding="utf-8",
)
from flowmorph_klein.visualization import make_contact_sheet
from PIL import Image

mainframe_images = [Image.open(record.path).convert("RGB") for record in MAINFRAME_RECORDS]
mainframe_contact_sheet = MAINFRAME_DIRECTORY / "mainframe_contact_sheet.png"
make_contact_sheet(
    mainframe_images,
    mainframe_contact_sheet,
    columns=5,
    labels=[record.id for record in MAINFRAME_RECORDS],
)
del mainframe_images
print(f"Prepared {len(MAINFRAME_RECORDS)} full-resolution keyframes in {MAINFRAME_DIRECTORY}")
MAINFRAME_DRIVE_REPORT = None
if DRIVE_ENABLED:
    MAINFRAME_DRIVE_REPORT = persist_artifact_tree(
        MAINFRAME_DIRECTORY,
        DRIVE_OUTPUT_ROOT,
        project_name=SPEC.project.name,
        label="mainframes",
    )
    print("Mainframes safely copied to Drive:", MAINFRAME_DRIVE_REPORT.destination)
    print({"files": MAINFRAME_DRIVE_REPORT.file_count, "bytes": MAINFRAME_DRIVE_REPORT.total_bytes})


In [ ]:
from IPython.display import Markdown, display
from PIL import Image

mainframe_sheet_preview = Image.open(mainframe_contact_sheet).convert("RGB")
mainframe_sheet_preview.thumbnail((CONTACT_SHEET_DISPLAY_MAX_WIDTH, 100000))
display(Markdown("### Generated keyframes — compact contact sheet"))
display(mainframe_sheet_preview)
print(f"Saved {len(MAINFRAME_RECORDS)} full-resolution keyframes and contact sheet to {MAINFRAME_DIRECTORY}")
del mainframe_sheet_preview


## 5. Fit and render scheduled FlowMorph transitions

This is the long-running section. Each selected leg performs the validated production-shape source fit, target fit, and twenty-frame render. The model download is cached, but the current audited runner loads and releases a fresh pipeline per transition. The output manifest records every literal prompt and its checksum.

In [ ]:
from flowmorph_klein.cli import select_hardware_profile
from flowmorph_klein.config import ProjectTemplateConfig, load_config, resolve_config
from flowmorph_klein.pipeline import FlowMorphRunner
import flowmorph_klein.renderer as flowmorph_renderer

# The reusable package defaults to a locked 512px research-reproduction contract.
# This notebook is explicitly experimental art mode, so retain dimensional
# safety while allowing the controls above to change resolution and fit settings.
def _validate_art_notebook_contract(config):
    for name, value in (("width", config.input.width), ("height", config.input.height)):
        if not 256 <= value <= 2048 or value % 16 != 0:
            raise ValueError(f"input.{name} must be 256–2048 and divisible by 16")

ProjectTemplateConfig._validate_full_shape_contract = _validate_art_notebook_contract
print("Experimental art controls enabled; research-only 512px parameter locks are inactive.")

def _art_alpha_values(frame_count):
    if frame_count < 2:
        raise ValueError("frame_count must be at least two")
    x = torch.linspace(0.0, 1.0, frame_count, dtype=torch.float64)
    curve = FLOWMORPH_ALPHA_CURVE.strip().lower()
    if curve == "linear":
        values = x
    elif curve == "smoothstep":
        values = x * x * (3.0 - 2.0 * x)
    elif curve == "smootherstep":
        # Quintic easing: zero velocity and zero acceleration at both endpoints.
        values = x * x * x * (x * (x * 6.0 - 15.0) + 10.0)
    elif curve == "sigmoid":
        if FLOWMORPH_ALPHA_SHARPNESS <= 0.0:
            raise ValueError("FLOWMORPH_ALPHA_SHARPNESS must be positive")
        values = torch.sigmoid((x - 0.5) * FLOWMORPH_ALPHA_SHARPNESS)
        values = (values - values[0]) / (values[-1] - values[0])
    else:
        raise ValueError("FLOWMORPH_ALPHA_CURVE must be linear, smoothstep, smootherstep, or sigmoid")
    return tuple(float(value) for value in values)

flowmorph_renderer.linear_alphas = _art_alpha_values
print("FlowMorph alpha curve:", FLOWMORPH_ALPHA_CURVE, _art_alpha_values(SPEC.flowmorph.frame_count))

record_by_id = {record.id: record for record in MAINFRAME_RECORDS}
selected_indices = (
    list(range(len(SPEC.transitions)))
    if TRANSITION_INDICES is None
    else list(TRANSITION_INDICES)
)
if any(index < 0 or index >= len(SPEC.transitions) for index in selected_indices):
    raise IndexError("TRANSITION_INDICES contains an invalid transition index")

TRANSITION_RUNS = []
for transition_index in selected_indices:
    transition = SPEC.transitions[transition_index]
    label = safe_transition_name(transition.from_id, transition.to_id)
    result_root = TRANSITION_ROOT / f"{transition_index:03d}_{label}"
    overrides = {
        "run_mode": "experimental",
        "project.name": f"{SPEC.project.name}_{label}",
        "model.id": SPEC.generation.model_id,
        "model.revision": SPEC.generation.model_revision,
        "lora.source": str(LOCAL_LORA_PATH),
        "lora.revision": None,
        "lora.weight_name": LOCAL_LORA_PATH.name,
        "lora.adapter_name": SPEC.lora.adapter_name,
        "lora.fit_scale": FLOWMORPH_FIT_LORA_SCALE,
        "lora.render_scale": FLOWMORPH_RENDER_LORA_SCALE,
        "lora.require_base_9b_compatibility": False,
        "lora.allow_distilled_9b": SPEC.lora.allow_distilled_9b_provenance,
        "input.source_image": str(record_by_id[transition.from_id].path),
        "input.target_image": str(record_by_id[transition.to_id].path),
        "input.source_prompt": record_by_id[transition.from_id].prompt,
        "input.target_prompt": record_by_id[transition.to_id].prompt,
        "input.bridge_prompt": None,
        "input.bridge_prompts": [
            apply_prompt_prefix(SPEC.generation.prompt_prefix, prompt)
            for prompt in transition.bridge_prompts
        ],
        "input.width": SPEC.generation.width,
        "input.height": SPEC.generation.height,
        "flowmorph.scheduler_points": FLOWMORPH_SCHEDULER_POINTS,
        "flowmorph.start_timestep_index": FLOWMORPH_START_TIMESTEP_INDEX,
        "flowmorph.optimization_steps_source": FLOWMORPH_SOURCE_OPTIMIZATION_STEPS,
        "flowmorph.optimization_steps_target": FLOWMORPH_TARGET_OPTIMIZATION_STEPS,
        "flowmorph.pred_learning_rate": FLOWMORPH_PRED_LEARNING_RATE,
        "flowmorph.u_learning_rate": FLOWMORPH_U_LEARNING_RATE,
        "flowmorph.render_indices": FLOWMORPH_RENDER_INDICES,
        "flowmorph.checkpoint_every": FLOWMORPH_CHECKPOINT_EVERY,
        "flowmorph.render_conditioning_mode": SPEC.flowmorph.render_conditioning_mode,
        "guidance.scale": FLOWMORPH_GUIDANCE_SCALE,
        "reproducibility.seed": SPEC.generation.seed + transition_index,
        "paths.input_root": str(MAINFRAME_DIRECTORY),
        "paths.work_root": str(PROJECT_OUTPUT / "work" / label),
        "paths.result_root": str(result_root),
        "paths.hf_cache": HF_CACHE_DIR,
        "paths.drive_root": None,
        "output.fps": SPEC.flowmorph.fps,
    }
    template = load_config(CONFIG_PATH, overrides=overrides)
    selected_profile = select_hardware_profile(PROFILE if PROFILE != "auto" else template.model.profile)
    config = resolve_config(template, selected_profile=selected_profile, check_input_files=True)
    print(f"[{transition_index + 1}/{len(SPEC.transitions)}] Starting {label}")
    runner = FlowMorphRunner.from_config(config)
    runner.prepare()
    runner.run_production_backward_probe()
    result = runner.run(resume=False)
    art_controls_path = Path(runner.run_directory) / "notebook_art_controls.json"
    art_controls_path.write_text(json.dumps({
        "alpha_curve": FLOWMORPH_ALPHA_CURVE,
        "alpha_sharpness": FLOWMORPH_ALPHA_SHARPNESS,
        "render_indices": list(FLOWMORPH_RENDER_INDICES),
        "mainframe_reference_strength": MAINFRAME_REFERENCE_STRENGTH,
        "mainframe_reference_blur": MAINFRAME_REFERENCE_BLUR,
        "flowmorph_guidance_scale": FLOWMORPH_GUIDANCE_SCALE,
        "fit_lora_scale": FLOWMORPH_FIT_LORA_SCALE,
        "render_lora_scale": FLOWMORPH_RENDER_LORA_SCALE,
    }, indent=2) + "\n", encoding="utf-8")
    drive_report = None
    if DRIVE_ENABLED:
        drive_report = persist_artifact_tree(
            runner.run_directory,
            DRIVE_OUTPUT_ROOT,
            project_name=SPEC.project.name,
            label=label,
        )
        print("Completed transition immediately copied to Drive:", drive_report.destination)
        print({"files": drive_report.file_count, "bytes": drive_report.total_bytes})
    TRANSITION_RUNS.append({
        "index": transition_index,
        "label": label,
        "run_directory": str(runner.run_directory),
        "archive": str(result.archive.path) if result.archive is not None else None,
        "drive_directory": str(drive_report.destination) if drive_report is not None else None,
    })
    display(Markdown(f"## Completed transition {transition_index + 1}: `{transition.from_id}` → `{transition.to_id}`"))
    print("Local run directory:", runner.run_directory)
    if drive_report is not None:
        print("Persistent Drive directory:", drive_report.destination)
    contact_sheet = Path(runner.run_directory) / "previews/display_contact_sheet.png"
    animated_preview = Path(runner.run_directory) / "previews/preview.gif"
    if contact_sheet.is_file():
        display(Markdown("**Contact sheet — all twenty ordered display frames**"))
        contact_sheet_preview = Image.open(contact_sheet).convert("RGB")
        contact_sheet_preview.thumbnail((CONTACT_SHEET_DISPLAY_MAX_WIDTH, 100000))
        display(contact_sheet_preview)
        del contact_sheet_preview
    if animated_preview.is_file():
        from IPython.display import Image as DisplayImage
        display(Markdown("**Animated transition preview**"))
        display(DisplayImage(filename=str(animated_preview), width=TRANSITION_PREVIEW_DISPLAY_WIDTH))
    display_frame_count = len(list(Path(runner.run_directory).joinpath("display_frames").glob("frame_*.png")))
    print(f"Saved {display_frame_count} full-resolution transition frames without embedding them in the notebook.")
    del result, runner, config, template
    gc.collect()
    torch.cuda.empty_cache()
print(TRANSITION_RUNS)


## 6. Assemble and audit the seamless closed loop

This runs only after all transitions have been rendered. The final transition target and opening display frame are checked against mainframe 1 pixel-for-pixel. Duplicate terminal endpoints are omitted so looping playback does not pause on two identical frames. With automatic rotation enabled, the export begins immediately after the quietest measured edge; disable it in section 1 to preserve nuclear physics as the first displayed frame. GIF and WebP loop internally, while MP4 playback must be set to repeat.

In [ ]:
import numpy as np
from flowmorph_klein.art_loop import collect_loop_frames
from flowmorph_klein.video import export_previews
from flowmorph_klein.visualization import make_contact_sheet
from PIL import Image

completed_indices = sorted(item["index"] for item in TRANSITION_RUNS)
if completed_indices != list(range(len(SPEC.transitions))):
    print("Partial test complete. Set TRANSITION_INDICES = None and rerun section 5 to assemble the full loop.")
else:
    ordered_runs = [
        next(item["run_directory"] for item in TRANSITION_RUNS if item["index"] == index)
        for index in range(len(SPEC.transitions))
    ]

    def _pixel_array(path):
        with Image.open(path) as opened:
            return np.asarray(opened.convert("RGB"), dtype=np.uint8)

    opening_display_path = Path(ordered_runs[0]) / "display_frames/frame_000.png"
    closing_display_path = (
        Path(ordered_runs[-1])
        / "display_frames"
        / f"frame_{SPEC.flowmorph.frame_count - 1:03d}.png"
    )
    canonical_first_path = MAINFRAME_RECORDS[0].path
    opening_is_exact = np.array_equal(_pixel_array(opening_display_path), _pixel_array(canonical_first_path))
    closing_is_exact = np.array_equal(_pixel_array(closing_display_path), _pixel_array(canonical_first_path))
    if not opening_is_exact or not closing_is_exact:
        raise RuntimeError(
            "Cyclic endpoint verification failed: the first display frame and final transition target "
            "must both be pixel-identical to mainframe 1."
        )

    loop_frames = collect_loop_frames(ordered_runs)
    # collect_loop_frames omits every duplicated target endpoint. The final stored
    # frame therefore leads into the exact first frame when the decoder wraps,
    # without displaying mainframe 1 twice and creating a one-frame pause.
    if LOOP_SEAM_AUDIT_SIZE < 32:
        raise ValueError("LOOP_SEAM_AUDIT_SIZE must be at least 32")

    def _metric_array(image):
        sample = image.convert("RGB").copy()
        sample.thumbnail((LOOP_SEAM_AUDIT_SIZE, LOOP_SEAM_AUDIT_SIZE))
        return np.asarray(sample, dtype=np.float32) / 255.0

    metric_frames = [_metric_array(frame) for frame in loop_frames]
    edge_scores = [
        float(np.mean(np.abs(metric_frames[index] - metric_frames[index - 1])))
        for index in range(len(metric_frames))
    ]
    quietest_cut_index = int(np.argmin(edge_scores))
    export_cut_index = quietest_cut_index if LOOP_AUTO_ROTATE_TO_QUIETEST_CUT else 0
    export_frames = loop_frames[export_cut_index:] + loop_frames[:export_cut_index]
    export_metric_frames = metric_frames[export_cut_index:] + metric_frames[:export_cut_index]

    incoming_motion = export_metric_frames[0] - export_metric_frames[-1]
    outgoing_motion = export_metric_frames[1] - export_metric_frames[0]
    seam_score = float(np.mean(np.abs(incoming_motion)))
    outgoing_score = float(np.mean(np.abs(outgoing_motion)))
    seam_motion_mismatch = float(np.mean(np.abs(outgoing_motion - incoming_motion)))
    median_edge_score = float(np.median(edge_scores))
    seam_ratio_to_median = seam_score / median_edge_score if median_edge_score else 0.0

    preview_directory = PROJECT_OUTPUT / "loop_previews"
    preview_paths = export_previews(
        export_frames, preview_directory, fps=SPEC.flowmorph.fps, hold_frames=0
    )
    seam_contact_sheet_path = preview_directory / "seam_audit.png"
    make_contact_sheet(
        [export_frames[-1], export_frames[0], export_frames[1]],
        seam_contact_sheet_path,
        columns=3,
        labels=["last before wrap", "exact playback start", "first after start"],
    )
    seam_audit = {
        "endpoint_pixels": {
            "opening_equals_mainframe_1": opening_is_exact,
            "closing_target_equals_mainframe_1": closing_is_exact,
        },
        "duplicate_terminal_frame_in_export": False,
        "canonical_frame_count": len(loop_frames),
        "export_frame_count": len(export_frames),
        "auto_rotate_enabled": LOOP_AUTO_ROTATE_TO_QUIETEST_CUT,
        "export_cut_index_in_canonical_sequence": export_cut_index,
        "quietest_cut_index_in_canonical_sequence": quietest_cut_index,
        "seam_mean_absolute_delta": seam_score,
        "first_outgoing_mean_absolute_delta": outgoing_score,
        "median_edge_mean_absolute_delta": median_edge_score,
        "seam_ratio_to_median_edge": seam_ratio_to_median,
        "seam_motion_mismatch": seam_motion_mismatch,
        "alpha_curve": FLOWMORPH_ALPHA_CURVE,
        "hold_frames": 0,
        "note": "The exact terminal copy of the opening frame is verified but intentionally omitted; looping playback supplies it without a duplicated-frame pause.",
    }
    seam_audit_path = preview_directory / "seam_audit.json"
    seam_audit_path.write_text(
        json.dumps(seam_audit, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
    )
    manifest = {
        "specification": SPEC.model_dump(mode="json", by_alias=True),
        "mainframes": [
            {"id": record.id, "seed": record.seed, "path": str(record.path)}
            for record in MAINFRAME_RECORDS
        ],
        "transition_runs": TRANSITION_RUNS,
        "previews": {name: str(path) for name, path in preview_paths.items()},
        "cyclic_export": seam_audit,
    }
    manifest_path = preview_directory / "art_loop_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    FINAL_DRIVE_REPORT = None
    if DRIVE_ENABLED:
        FINAL_DRIVE_REPORT = persist_artifact_tree(
            preview_directory,
            DRIVE_OUTPUT_ROOT,
            project_name=SPEC.project.name,
            label="assembled_loop",
        )
        print("Assembled loop immediately copied to Drive:", FINAL_DRIVE_REPORT.destination)
    print(f"Assembled {len(export_frames)} duplicate-free cyclic frames; manifest saved to {manifest_path}")
    print({
        "pixel_exact_closure": closing_is_exact,
        "cut_index": export_cut_index,
        "seam_vs_median_edge": round(seam_ratio_to_median, 4),
        "motion_mismatch": round(seam_motion_mismatch, 6),
    })
    from IPython.display import Image as DisplayImage
    seam_preview = Image.open(seam_contact_sheet_path).convert("RGB")
    seam_preview.thumbnail((LOOP_SEAM_DISPLAY_MAX_WIDTH, 100000))
    display(Markdown("## Loop seam audit: last → exact start → next"))
    display(seam_preview)
    del seam_preview
    display(Markdown("## Complete closed-loop preview"))
    display(DisplayImage(filename=str(preview_paths["gif"]), width=TRANSITION_PREVIEW_DISPLAY_WIDTH))
    if DOWNLOAD_FINAL_PREVIEW:
        try:
            from google.colab import files
        except ImportError:
            print("Colab download helper unavailable; use the printed preview path.")
        else:
            files.download(str(preview_paths["mp4"]))


## 7. RIFE interpolation and circular SSIM motion resampling

This optional finishing stage reconstructs the useful part of the older interpolation project. It applies RIFE directly to the lossless cyclic PNG sequence, including the last-to-first pair, then redistributes a smaller set of frames at equal increments of cumulative perceptual motion measured by `1 − SSIM`. Unlike the old implementation, the SSIM calculation includes the wraparound edge and selection is constrained to unique ordered frames. Temporary dense PNGs are excluded from Drive persistence and may be deleted after a successful encode.

In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

if not RUN_RIFE_POSTPROCESS:
    print("RIFE post-processing disabled in section 1.")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("RIFE post-processing requires the CUDA runtime used above.")
    if RIFE_MULTIPLIER < 2:
        raise ValueError("RIFE_MULTIPLIER must be at least 2")
    if RIFE_SCALE not in {0.25, 0.5, 1.0, 2.0, 4.0}:
        raise ValueError("RIFE_SCALE must be one of 0.25, 0.5, 1.0, 2.0, or 4.0")
    if RIFE_FINAL_FPS <= 0:
        raise ValueError("RIFE_FINAL_FPS must be positive")
    if RIFE_SSIM_ANALYSIS_SIZE < 32:
        raise ValueError("RIFE_SSIM_ANALYSIS_SIZE must be at least 32")

    rife_root = Path(RIFE_ROOT)
    if not (rife_root / ".git").is_dir():
        if rife_root.exists():
            raise RuntimeError(f"RIFE_ROOT exists but is not a Git checkout: {rife_root}")
        subprocess.check_call([
            "git", "clone", "--filter=blob:none", RIFE_REPOSITORY_URL, str(rife_root)
        ])
    installed_rife_revision = subprocess.check_output(
        ["git", "-C", str(rife_root), "rev-parse", "HEAD"], text=True
    ).strip()
    if installed_rife_revision != RIFE_REPOSITORY_REVISION:
        subprocess.check_call([
            "git", "-C", str(rife_root), "fetch", "--depth", "1",
            "origin", RIFE_REPOSITORY_REVISION,
        ])
        subprocess.check_call([
            "git", "-C", str(rife_root), "checkout", "--detach", RIFE_REPOSITORY_REVISION
        ])
    installed_rife_revision = subprocess.check_output(
        ["git", "-C", str(rife_root), "rev-parse", "HEAD"], text=True
    ).strip()
    if installed_rife_revision != RIFE_REPOSITORY_REVISION:
        raise RuntimeError("Practical-RIFE checkout did not resolve to the pinned revision")

    from huggingface_hub import hf_hub_download
    rife_model_archive = Path(hf_hub_download(
        repo_id=RIFE_MODEL_REPOSITORY,
        filename=RIFE_MODEL_FILENAME,
        revision=RIFE_MODEL_REVISION,
        cache_dir=HF_CACHE_DIR,
    ))
    rife_model_root = Path(HF_CACHE_DIR) / "flowmorph_rife_models" / RIFE_MODEL_FILENAME.removesuffix(".zip")
    flownet_candidates = list(rife_model_root.rglob("flownet.pkl")) if rife_model_root.exists() else []
    if not flownet_candidates:
        rife_model_root.mkdir(parents=True, exist_ok=True)
        root_resolved = rife_model_root.resolve()
        with zipfile.ZipFile(rife_model_archive) as archive:
            for member in archive.infolist():
                destination = (rife_model_root / member.filename).resolve()
                if not destination.is_relative_to(root_resolved):
                    raise RuntimeError(f"Unsafe path in RIFE model archive: {member.filename}")
            archive.extractall(rife_model_root)
        flownet_candidates = list(rife_model_root.rglob("flownet.pkl"))
    if len(flownet_candidates) != 1:
        raise RuntimeError(f"Expected one RIFE flownet.pkl, found {flownet_candidates}")
    RIFE_MODEL_DIRECTORY = flownet_candidates[0].parent
    for required_name in ("IFNet_HDv3.py", "flownet.pkl"):
        if not (RIFE_MODEL_DIRECTORY / required_name).is_file():
            raise FileNotFoundError(RIFE_MODEL_DIRECTORY / required_name)
    print({
        "rife_revision": installed_rife_revision,
        "model": RIFE_MODEL_FILENAME,
        "model_directory": str(RIFE_MODEL_DIRECTORY),
        "multiplier": RIFE_MULTIPLIER,
        "scale": RIFE_SCALE,
        "fp16": RIFE_USE_FP16,
    })

    RIFE_RUNNER_SOURCE = r'''
import argparse
import shutil
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

parser = argparse.ArgumentParser()
parser.add_argument("--repo", required=True)
parser.add_argument("--model", required=True)
parser.add_argument("--input", required=True)
parser.add_argument("--output", required=True)
parser.add_argument("--multi", type=int, required=True)
parser.add_argument("--scale", type=float, required=True)
parser.add_argument("--fp16", action="store_true")
args = parser.parse_args()

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this RIFE finishing pass")
sys.path.insert(0, str(Path(args.model).resolve().parent))
sys.path.insert(0, str(Path(args.repo).resolve()))
from train_log.IFNet_HDv3 import IFNet

input_paths = sorted(Path(args.input).glob("*.png"), key=lambda path: int(path.stem))
if len(input_paths) < 2:
    raise ValueError("RIFE input needs at least two numbered PNG files")
output = Path(args.output)
output.mkdir(parents=True, exist_ok=False)

class InferenceModel:
    def __init__(self, model_directory):
        self.flownet = IFNet().to("cuda")
        state = torch.load(
            str(Path(model_directory) / "flownet.pkl"), map_location="cpu", weights_only=True
        )
        state = {
            (key.removeprefix("module.")): value for key, value in state.items()
        }
        load_result = self.flownet.load_state_dict(state, strict=False)
        if load_result.missing_keys:
            raise RuntimeError(f"RIFE checkpoint is missing inference keys: {load_result.missing_keys}")
        self.flownet.eval()

    def inference(self, image0, image1, timestep, scale):
        inputs = torch.cat((image0, image1), dim=1)
        scale_list = [16 / scale, 8 / scale, 4 / scale, 2 / scale, 1 / scale]
        _, _, merged = self.flownet(inputs, timestep, scale_list)
        return merged[-1]

model = InferenceModel(args.model)
if args.fp16:
    model.flownet.half()

device = torch.device("cuda")
first_image = Image.open(input_paths[0]).convert("RGB")
height, width = first_image.height, first_image.width
block = max(128, int(128 / args.scale))
padded_height = ((height - 1) // block + 1) * block
padded_width = ((width - 1) // block + 1) * block
padding = (0, padded_width - width, 0, padded_height - height)

def load_tensor(path):
    image = Image.open(path).convert("RGB")
    if image.size != (width, height):
        raise ValueError(f"Mismatched RIFE input dimensions at {path}: {image.size}")
    array = np.asarray(image, dtype=np.uint8).copy()
    tensor = torch.from_numpy(array.transpose(2, 0, 1)).unsqueeze(0).to(device).float() / 255.0
    if args.fp16:
        tensor = tensor.half()
    return F.pad(tensor, padding)

def save_tensor(tensor, path):
    array = (tensor[0, :, :height, :width].float().clamp(0, 1) * 255.0).round().byte()
    array = array.permute(1, 2, 0).cpu().numpy()
    Image.fromarray(array, mode="RGB").save(path, compress_level=4)

output_index = 0
shutil.copy2(input_paths[0], output / f"{output_index:07d}.png")
output_index += 1
report_every = max(1, (len(input_paths) - 1) // 20)
with torch.inference_mode():
    left = load_tensor(input_paths[0])
    for pair_index, right_path in enumerate(input_paths[1:], start=1):
        right = load_tensor(right_path)
        for step in range(1, args.multi):
            timestep = step / args.multi
            middle = model.inference(left, right, timestep=timestep, scale=args.scale)
            save_tensor(middle, output / f"{output_index:07d}.png")
            output_index += 1
        shutil.copy2(right_path, output / f"{output_index:07d}.png")
        output_index += 1
        left = right
        if pair_index % report_every == 0 or pair_index == len(input_paths) - 1:
            print(f"RIFE pairs: {pair_index}/{len(input_paths) - 1}; frames: {output_index}", flush=True)
print(f"RIFE complete: {output_index} PNG frames")
'''
    RIFE_RUNNER_PATH = Path(PROJECT_OUTPUT) / "rife_pair_sequence_runner.py"
    RIFE_RUNNER_PATH.write_text(RIFE_RUNNER_SOURCE.strip() + "\n", encoding="utf-8")
    print("Pinned Practical-RIFE and model are ready; isolated pair-sequence runner written to:", RIFE_RUNNER_PATH)


In [ ]:
import shutil
from datetime import datetime, timezone

if RUN_RIFE_POSTPROCESS:
    if "export_frames" not in globals():
        raise RuntimeError("Run the complete-loop assembly cell before RIFE post-processing.")
    postprocess_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    RIFE_WORK_DIRECTORY = Path(PROJECT_OUTPUT) / "rife_work" / postprocess_stamp
    RIFE_INPUT_DIRECTORY = RIFE_WORK_DIRECTORY / "cyclic_input"
    RIFE_DENSE_DIRECTORY = RIFE_WORK_DIRECTORY / "dense_frames"
    RIFE_RESULTS_DIRECTORY = Path(PROJECT_OUTPUT) / "postprocessed_video" / postprocess_stamp
    RIFE_INPUT_DIRECTORY.mkdir(parents=True, exist_ok=False)
    RIFE_RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=False)

    for index, frame in enumerate(export_frames):
        frame.convert("RGB").save(
            RIFE_INPUT_DIRECTORY / f"{index:07d}.png", compress_level=4
        )
    # Append the exact opening image once so RIFE also interpolates the wraparound pair.
    shutil.copy2(
        RIFE_INPUT_DIRECTORY / "0000000.png",
        RIFE_INPUT_DIRECTORY / f"{len(export_frames):07d}.png",
    )
    rife_command = [
        sys.executable, str(RIFE_RUNNER_PATH),
        "--repo", str(rife_root),
        "--model", str(RIFE_MODEL_DIRECTORY),
        "--input", str(RIFE_INPUT_DIRECTORY),
        "--output", str(RIFE_DENSE_DIRECTORY),
        "--multi", str(RIFE_MULTIPLIER),
        "--scale", str(RIFE_SCALE),
    ]
    if RIFE_USE_FP16:
        rife_command.append("--fp16")
    print(f"Interpolating {len(export_frames)} cyclic frames at {RIFE_MULTIPLIER}× density...")
    subprocess.check_call(rife_command)

    dense_paths_with_duplicate = sorted(
        RIFE_DENSE_DIRECTORY.glob("*.png"), key=lambda path: int(path.stem)
    )
    expected_dense_count = len(export_frames) * RIFE_MULTIPLIER + 1
    if len(dense_paths_with_duplicate) != expected_dense_count:
        raise RuntimeError(
            f"RIFE wrote {len(dense_paths_with_duplicate)} frames; expected {expected_dense_count}"
        )
    with Image.open(dense_paths_with_duplicate[0]) as first_dense:
        first_dense_array = np.asarray(first_dense.convert("RGB"))
    with Image.open(dense_paths_with_duplicate[-1]) as last_dense:
        last_dense_array = np.asarray(last_dense.convert("RGB"))
    if not np.array_equal(first_dense_array, last_dense_array):
        raise RuntimeError("RIFE cyclic terminal frame is not pixel-identical to its opening frame")
    RIFE_DENSE_PATHS = dense_paths_with_duplicate[:-1]
    print({
        "base_cyclic_frames": len(export_frames),
        "dense_unique_frames": len(RIFE_DENSE_PATHS),
        "removed_exact_terminal_duplicate": True,
        "work_directory": str(RIFE_WORK_DIRECTORY),
    })


In [ ]:
import json
import math
import os
import shutil
import subprocess

if RUN_RIFE_POSTPROCESS:
    from skimage.metrics import structural_similarity
    import imageio_ffmpeg
    import matplotlib.pyplot as plt
    from IPython.display import Video

    def _ssim_luma(path):
        with Image.open(path) as opened:
            gray = opened.convert("L")
            gray.thumbnail((RIFE_SSIM_ANALYSIS_SIZE, RIFE_SSIM_ANALYSIS_SIZE))
            return np.asarray(gray, dtype=np.uint8)

    print(f"Computing circular SSIM motion weights for {len(RIFE_DENSE_PATHS)} dense frames...")
    dense_luma = [_ssim_luma(path) for path in RIFE_DENSE_PATHS]
    circular_ssim = np.asarray([
        structural_similarity(dense_luma[index - 1], dense_luma[index], data_range=255)
        for index in range(len(dense_luma))
    ], dtype=np.float64)
    motion_weights = np.maximum(RIFE_SSIM_WEIGHT_FLOOR, 1.0 - circular_ssim)
    frame_positions = np.zeros(len(RIFE_DENSE_PATHS), dtype=np.float64)
    frame_positions[1:] = np.cumsum(motion_weights[1:])
    total_circular_motion = float(frame_positions[-1] + motion_weights[0])

    canonical_duration = len(export_frames) / float(SPEC.flowmorph.fps)
    target_frame_count = int(round(canonical_duration * RIFE_FINAL_FPS))
    if target_frame_count < 3:
        raise ValueError("RIFE final video needs at least three frames")
    if target_frame_count > len(RIFE_DENSE_PATHS):
        raise ValueError(
            f"Requested {target_frame_count} final frames from only {len(RIFE_DENSE_PATHS)} dense frames; "
            "increase RIFE_MULTIPLIER or lower RIFE_FINAL_FPS"
        )
    target_positions = np.linspace(0.0, total_circular_motion, target_frame_count, endpoint=False)

    # Nearest cumulative-motion sample with strict monotonic bounds. This preserves
    # duration without repeating frames, unlike the old np.searchsorted-only method.
    selected_indices = []
    previous_index = -1
    dense_count = len(RIFE_DENSE_PATHS)
    for order, target in enumerate(target_positions):
        minimum_index = previous_index + 1
        maximum_index = dense_count - (target_frame_count - order)
        insertion = int(np.searchsorted(frame_positions, target, side="left"))
        candidates = {
            min(max(insertion, minimum_index), maximum_index),
            min(max(insertion - 1, minimum_index), maximum_index),
        }
        chosen = min(candidates, key=lambda index: abs(frame_positions[index] - target))
        selected_indices.append(chosen)
        previous_index = chosen
    selected_indices[0] = 0
    selected_indices[-1] = dense_count - 1
    if len(set(selected_indices)) != target_frame_count:
        raise RuntimeError("SSIM motion resampling produced duplicate frame selections")

    selected_directory = RIFE_WORK_DIRECTORY / "ssim_resampled_frames"
    selected_directory.mkdir(parents=True, exist_ok=False)
    for output_index, dense_index in enumerate(selected_indices):
        source = RIFE_DENSE_PATHS[dense_index]
        destination = selected_directory / f"{output_index:07d}.png"
        try:
            os.link(source, destination)
        except OSError:
            shutil.copy2(source, destination)

    RIFE_FINAL_VIDEO_PATH = RIFE_RESULTS_DIRECTORY / "flowmorph_rife_ssim_loop.mp4"
    ffmpeg_executable = imageio_ffmpeg.get_ffmpeg_exe()
    ffmpeg_command = [
        ffmpeg_executable, "-y",
        "-framerate", str(RIFE_FINAL_FPS),
        "-i", str(selected_directory / "%07d.png"),
        "-an", "-c:v", "libx264",
        "-preset", "slow", "-crf", str(RIFE_VIDEO_CRF),
        "-pix_fmt", "yuv420p", "-movflags", "+faststart",
        str(RIFE_FINAL_VIDEO_PATH),
    ]
    subprocess.check_call(ffmpeg_command)
    if not RIFE_FINAL_VIDEO_PATH.is_file() or RIFE_FINAL_VIDEO_PATH.stat().st_size == 0:
        raise RuntimeError("FFmpeg did not create the final RIFE/SSIM MP4")

    selected_ssim = np.asarray([
        structural_similarity(
            dense_luma[selected_indices[index - 1]],
            dense_luma[selected_indices[index]],
            data_range=255,
        )
        for index in range(target_frame_count)
    ], dtype=np.float64)
    selected_motion = 1.0 - selected_ssim
    postprocess_report = {
        "method": "lossless cyclic PNGs -> Practical-RIFE pair interpolation -> circular 1-SSIM arc-length resampling -> H.264 MP4",
        "rife_repository": RIFE_REPOSITORY_URL,
        "rife_revision": RIFE_REPOSITORY_REVISION,
        "rife_model_repository": RIFE_MODEL_REPOSITORY,
        "rife_model_revision": RIFE_MODEL_REVISION,
        "rife_model_filename": RIFE_MODEL_FILENAME,
        "rife_multiplier": RIFE_MULTIPLIER,
        "rife_scale": RIFE_SCALE,
        "rife_fp16": RIFE_USE_FP16,
        "base_frames": len(export_frames),
        "dense_unique_frames": len(RIFE_DENSE_PATHS),
        "final_unique_frames": target_frame_count,
        "base_fps": float(SPEC.flowmorph.fps),
        "final_fps": RIFE_FINAL_FPS,
        "duration_seconds": target_frame_count / RIFE_FINAL_FPS,
        "circular_motion_total": total_circular_motion,
        "dense_ssim": {
            "mean": float(circular_ssim.mean()),
            "median": float(np.median(circular_ssim)),
            "minimum": float(circular_ssim.min()),
            "wraparound": float(circular_ssim[0]),
        },
        "resampled_ssim": {
            "mean": float(selected_ssim.mean()),
            "median": float(np.median(selected_ssim)),
            "minimum": float(selected_ssim.min()),
            "wraparound": float(selected_ssim[0]),
            "motion_coefficient_of_variation": (
                float(selected_motion.std() / selected_motion.mean())
                if selected_motion.mean() else 0.0
            ),
        },
        "selected_dense_indices": selected_indices,
        "terminal_duplicate_in_video": False,
        "video": str(RIFE_FINAL_VIDEO_PATH),
    }
    RIFE_REPORT_PATH = RIFE_RESULTS_DIRECTORY / "rife_ssim_report.json"
    RIFE_REPORT_PATH.write_text(
        json.dumps(postprocess_report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
    )

    RIFE_SSIM_PLOT_PATH = RIFE_RESULTS_DIRECTORY / "ssim_motion_profile.png"
    figure, axes = plt.subplots(2, 1, figsize=(12, 5), constrained_layout=True)
    axes[0].plot(motion_weights, linewidth=0.7, color="#4b6a88")
    axes[0].scatter([0], [motion_weights[0]], color="#c43d32", s=24, label="wrap edge")
    axes[0].set_title("Dense RIFE motion profile (1 − circular SSIM)")
    axes[0].legend(loc="upper right")
    axes[1].plot(selected_motion, linewidth=0.8, color="#7a5535")
    axes[1].scatter([0], [selected_motion[0]], color="#c43d32", s=24, label="wrap edge")
    axes[1].set_title("After equal-motion resampling")
    axes[1].set_xlabel("Frame edge")
    axes[1].legend(loc="upper right")
    for axis in axes:
        axis.set_ylabel("1 − SSIM")
        axis.grid(alpha=0.2)
    figure.savefig(RIFE_SSIM_PLOT_PATH, dpi=160, facecolor="white")
    plt.close(figure)

    RIFE_DRIVE_REPORT = None
    if DRIVE_ENABLED:
        RIFE_DRIVE_REPORT = persist_artifact_tree(
            RIFE_RESULTS_DIRECTORY,
            DRIVE_OUTPUT_ROOT,
            project_name=SPEC.project.name,
            label="rife_ssim_final",
        )
        print("Final RIFE/SSIM video immediately copied to Drive:", RIFE_DRIVE_REPORT.destination)

    motion_plot_preview = Image.open(RIFE_SSIM_PLOT_PATH).convert("RGB")
    motion_plot_preview.thumbnail((CONTACT_SHEET_DISPLAY_MAX_WIDTH, 100000))
    display(Markdown("## Final RIFE + circular SSIM diagnostics"))
    display(motion_plot_preview)
    del motion_plot_preview
    display(Video(
        filename=str(RIFE_FINAL_VIDEO_PATH),
        embed=False,
        width=RIFE_DISPLAY_WIDTH,
        html_attributes="controls loop muted playsinline",
    ))
    print({
        "final_video": str(RIFE_FINAL_VIDEO_PATH),
        "frames": target_frame_count,
        "fps": RIFE_FINAL_FPS,
        "seconds": round(target_frame_count / RIFE_FINAL_FPS, 3),
        "wraparound_ssim": round(float(selected_ssim[0]), 6),
        "motion_variation": round(postprocess_report["resampled_ssim"]["motion_coefficient_of_variation"], 6),
    })

    if DOWNLOAD_FINAL_PREVIEW:
        try:
            from google.colab import files
        except ImportError:
            print("Colab download helper unavailable; use the final video path above.")
        else:
            files.download(str(RIFE_FINAL_VIDEO_PATH))

    if not RIFE_KEEP_WORK_FRAMES:
        expected_work_parent = (Path(PROJECT_OUTPUT) / "rife_work").resolve()
        work_to_remove = RIFE_WORK_DIRECTORY.resolve()
        if work_to_remove.parent != expected_work_parent:
            raise RuntimeError(f"Refusing to remove unexpected RIFE work directory: {work_to_remove}")
        shutil.rmtree(work_to_remove)
        print("Removed temporary dense RIFE PNGs after successful video and report creation.")
